# ROGII — Wellbore Geology Prediction
## Notebook 3: Advanced Techniques, Hyperparameter Tuning & Exotic Approaches

This notebook goes beyond the baseline ensemble and covers:

| Section | Technique | Expected LB gain |
|---|---|---|
| 1 | Advanced Feature Engineering | -1 to -2 RMSE |
| 2 | Optuna Hyperparameter Tuning | -0.5 to -1 RMSE |
| 3 | Stacking Meta-Learner | -0.3 to -0.7 RMSE |
| 4 | LSTM Sequence Model | -0.5 to -1.5 RMSE |
| 5 | Particle Filter Post-Processing | -0.5 to -2 RMSE |
| 6 | Pseudo-Labeling | -0.2 to -0.5 RMSE |
| 7 | Reference Well Correlation | -0.5 to -1 RMSE |
| 8 | Test-Time Augmentation (TTA) | -0.1 to -0.3 RMSE |

> **Note**: gains are additive only if techniques are independent. In practice, expect ~60% of the sum.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge, HuberRegressor
from sklearn.preprocessing import StandardScaler
import scipy.signal as signal
import pywt

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('husl')

DATA_DIR   = Path('../data')
SUBMIT_DIR = Path('../submissions')
SUBMIT_DIR.mkdir(exist_ok=True)

SEED    = 42
N_FOLDS = 5

# Column names — adjust if needed
WELL_COL   = 'well_id'
TARGET_COL = 'TVT'
DEPTH_COL  = 'DEPT'
GR_COL     = 'GR'
X_COL, Y_COL, Z_COL = 'X', 'Y', 'Z'

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f'Train: {train.shape}  |  Test: {test.shape}')

---
# PART 1 — Advanced Feature Engineering

We expand far beyond rolling stats and basic DWT.

## 1.1 Fourier Transform Features

**What**: Apply FFT to the GR log within a sliding window and extract dominant frequencies and their amplitudes.

**Why it works**: Geology has periodic layering (cyclothems). FFT captures the periodicity of sandstone/shale alternation, which is directly related to TVT. DWT is local; FFT is global within the window.

**When to use**: When wells are long (>200 samples). Short wells lose FFT resolution.

In [ ]:
def fft_features(series, window=64, top_k=5):
    """
    Sliding-window FFT: at each position, compute FFT over the preceding
    `window` samples and extract the top-k frequency amplitudes and their indices.
    Returns a DataFrame with 2*top_k columns per position.
    """
    n = len(series)
    vals = series.values
    # Pad left so we have a full window from the start
    padded = np.pad(vals, (window - 1, 0), mode='edge')
    
    amp_cols = {f'fft_amp_{k}': np.zeros(n) for k in range(top_k)}
    freq_cols = {f'fft_freq_{k}': np.zeros(n) for k in range(top_k)}
    
    for i in range(n):
        chunk = padded[i:i + window]
        # Detrend to remove slow drift
        chunk = chunk - np.polyval(np.polyfit(np.arange(window), chunk, 1),
                                    np.arange(window))
        # Apply Hanning window to reduce spectral leakage
        chunk = chunk * np.hanning(window)
        spectrum = np.abs(np.fft.rfft(chunk))
        freqs    = np.fft.rfftfreq(window)
        # Top-k by amplitude (skip DC component at freq=0)
        top_idx = np.argsort(spectrum[1:])[-top_k:][::-1] + 1
        for k, idx in enumerate(top_idx):
            amp_cols[f'fft_amp_{k}'][i]  = spectrum[idx]
            freq_cols[f'fft_freq_{k}'][i] = freqs[idx]
    
    result = {**amp_cols, **freq_cols}
    return pd.DataFrame(result, index=series.index)


# Demo: visualize FFT spectrum on one well
sample_well = train[WELL_COL].unique()[0]
well_gr = train[train[WELL_COL] == sample_well].sort_values(DEPTH_COL)[GR_COL].dropna()

spectrum = np.abs(np.fft.rfft(well_gr.values - well_gr.mean()))
freqs    = np.fft.rfftfreq(len(well_gr))

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].plot(well_gr.values, lw=0.8, color='darkorange')
axes[0].set_title(f'GR log — Well {sample_well}')
axes[0].set_xlabel('Depth index')
axes[0].set_ylabel('GR (API)')

axes[1].plot(freqs[1:], spectrum[1:], lw=1)
axes[1].set_title('FFT Amplitude Spectrum of GR log')
axes[1].set_xlabel('Frequency (cycles/sample)')
axes[1].set_ylabel('Amplitude')
dominant_f = freqs[1:][np.argmax(spectrum[1:])]
axes[1].axvline(dominant_f, color='red', ls='--',
                label=f'Dominant freq: {dominant_f:.4f} → period ≈ {1/dominant_f:.0f} samples')
axes[1].legend()

plt.tight_layout()
plt.show()

## 1.2 Exponential Moving Average (EMA) + MACD

**What**: Exponential moving averages weight recent samples more. MACD (Moving Average Convergence/Divergence) is the difference between a fast and slow EMA — borrowed from financial signal processing.

**Why it works**: TVT transitions (entering/leaving a layer) create "momentum" in the GR signal. MACD detects when the short-term GR trend diverges from the long-term baseline — exactly what happens at geological boundaries.

**Signal analogy**: MACD is like a first derivative but smoother and more physically interpretable.

In [ ]:
def ema_macd_features(series):
    """
    Computes multiple EMA spans and MACD-style crossover signals.
    
    MACD(fast, slow) = EMA(fast) - EMA(slow)
    Signal line      = EMA(MACD, 9)  — like a smoothed MACD
    Histogram        = MACD - Signal — momentum acceleration
    """
    features = {}
    
    spans = [3, 5, 10, 20, 50]
    emas = {}
    for s in spans:
        emas[s] = series.ewm(span=s, adjust=False).mean()
        features[f'ema_{s}'] = emas[s]
    
    # MACD pairs: (fast, slow)
    for fast, slow in [(3, 10), (5, 20), (10, 50)]:
        macd = emas[fast] - emas[slow]
        sig  = macd.ewm(span=9, adjust=False).mean()
        hist = macd - sig
        features[f'macd_{fast}_{slow}']     = macd
        features[f'macd_sig_{fast}_{slow}'] = sig
        features[f'macd_hist_{fast}_{slow}'] = hist
    
    return pd.DataFrame(features, index=series.index)


# Visualize MACD on one well
well_df = train[train[WELL_COL] == sample_well].sort_values(DEPTH_COL)
gr_norm = (well_df[GR_COL] - well_df[GR_COL].mean()) / (well_df[GR_COL].std() + 1e-6)
gr_norm.index = range(len(gr_norm))

macd_df = ema_macd_features(gr_norm)

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

axes[0].plot(gr_norm.values, lw=0.8, color='grey', label='GR norm')
axes[0].plot(macd_df['ema_5'].values,  lw=1.2, label='EMA-5 (fast)')
axes[0].plot(macd_df['ema_20'].values, lw=1.2, label='EMA-20 (slow)')
axes[0].set_title('GR log with EMA overlays')
axes[0].legend()

macd_vals = macd_df['macd_5_20'].values
sig_vals  = macd_df['macd_sig_5_20'].values
axes[1].plot(macd_vals, lw=1, label='MACD(5,20)', color='steelblue')
axes[1].plot(sig_vals,  lw=1, label='Signal line', color='orange')
axes[1].axhline(0, color='black', lw=0.5)
axes[1].set_title('MACD — detects GR trend momentum (geological boundary proximity)')
axes[1].legend()

hist = macd_df['macd_hist_5_20'].values
colors = ['green' if v >= 0 else 'red' for v in hist]
axes[2].bar(range(len(hist)), hist, color=colors, alpha=0.7)
axes[2].set_title('MACD Histogram — momentum acceleration (spikes = layer transitions)')
axes[2].set_xlabel('Depth index')

plt.tight_layout()
plt.show()

## 1.3 Higher-Order Statistical Moments

**What**: Rolling skewness and kurtosis of GR within a window.

**Why it works**: 
- **Skewness** measures asymmetry of the GR distribution in the window. Near the top of a sand (entry from shale), GR is right-skewed; near the bottom (entry to shale below), left-skewed.
- **Kurtosis** measures the "peakedness". A bimodal GR window (sand + shale alternation) has low kurtosis; a uniform lithology has high kurtosis.

Both encode where in the reservoir layer the well is, independent of the absolute GR level.

In [ ]:
from scipy.stats import skew, kurtosis

def statistical_moment_features(series, windows=(10, 20, 50)):
    features = {}
    for w in windows:
        r = series.rolling(w, min_periods=max(4, w // 4))
        features[f'roll_skew_{w}']  = r.skew()
        features[f'roll_kurt_{w}']  = r.kurt()
        features[f'roll_cv_{w}']    = r.std() / (r.mean().abs() + 1e-6)  # coefficient of variation
        # Quantile range (robust spread)
        features[f'roll_iqr_{w}']   = r.quantile(0.75) - r.quantile(0.25)
        # % samples above median (asymmetry measure)
        med = r.median()
        features[f'roll_above_med_{w}'] = r.apply(lambda x: (x > x.median()).mean())
    return pd.DataFrame(features, index=series.index)

## 1.4 Reference Well Cross-Correlation

**What**: Cross-correlate the horizontal well's GR log with the reference (vertical) well's GR log at multiple depth offsets. Find the lag that maximizes correlation and use it as a feature.

**Why it works**: The reference (type) well provides the expected GR pattern for the same geological formation in the area. If the horizontal well is 5m above the reference well's trajectory, the GR logs will be offset by the TVT difference. The cross-correlation lag is essentially a direct estimate of TVT!

**This is the single most powerful domain feature** — it's essentially solving the problem with signal processing before any ML.

In [ ]:
def xcorr_tvt_estimate(horizontal_gr, reference_gr, max_lag=50):
    """
    Estimates TVT at each depth point by cross-correlating the horizontal GR
    with the reference well GR.
    
    Returns:
      - best_lag: lag at peak cross-correlation (in depth-sample units → convert to meters)
      - best_corr: peak correlation value (confidence)
      - corr_curve: full correlation array (can be used as features)
    """
    h = horizontal_gr - horizontal_gr.mean()
    r = reference_gr  - reference_gr.mean()
    
    lags   = np.arange(-max_lag, max_lag + 1)
    corrs  = np.zeros(len(lags))
    
    h_std = h.std() + 1e-6
    r_std = r.std() + 1e-6
    
    for i, lag in enumerate(lags):
        if lag >= 0:
            h_slice = h[lag:]
            r_slice = r[:len(h_slice)]
        else:
            r_slice = r[-lag:]
            h_slice = h[:len(r_slice)]
        
        if len(h_slice) < 10:
            continue
        corrs[i] = np.corrcoef(h_slice, r_slice)[0, 1]
    
    best_i    = np.argmax(corrs)
    best_lag  = lags[best_i]
    best_corr = corrs[best_i]
    
    return best_lag, best_corr, corrs


def sliding_xcorr_features(df, well_col, depth_col, gr_col, ref_gr_col, window=100, step=1):
    """
    For each well, compute sliding-window cross-correlation between the horizontal
    well GR and the reference well GR.
    
    `ref_gr_col`: column containing the reference well's GR value at the corresponding depth.
    This column must already be aligned/resampled to the horizontal well's depth grid.
    """
    all_parts = []
    
    for well_id, gdf in df.groupby(well_col):
        gdf = gdf.sort_values(depth_col).copy()
        n = len(gdf)
        
        best_lags  = np.full(n, np.nan)
        best_corrs = np.full(n, np.nan)
        
        if ref_gr_col not in gdf.columns or gdf[ref_gr_col].isnull().all():
            gdf['xcorr_lag']  = np.nan
            gdf['xcorr_conf'] = np.nan
            all_parts.append(gdf)
            continue
        
        h_gr = gdf[gr_col].fillna(gdf[gr_col].median())
        r_gr = gdf[ref_gr_col].fillna(gdf[ref_gr_col].median())
        
        for i in range(n):
            lo = max(0, i - window // 2)
            hi = min(n, i + window // 2)
            if hi - lo < 20:
                continue
            lag, corr, _ = xcorr_tvt_estimate(h_gr.iloc[lo:hi], r_gr.iloc[lo:hi])
            best_lags[i]  = lag
            best_corrs[i] = corr
        
        gdf['xcorr_lag']  = best_lags
        gdf['xcorr_conf'] = best_corrs
        all_parts.append(gdf)
    
    return pd.concat(all_parts, ignore_index=True)


print('Cross-correlation feature functions defined.')
print('Requires a ref_gr column (reference/type well GR resampled to horizontal well depth grid).')

## 1.5 Full Advanced Feature Engineering Pipeline

In [ ]:
def compute_tortuosity(df, x_col, y_col, z_col, window=10):
    dx = df[x_col].diff().fillna(0)
    dy = df[y_col].diff().fillna(0)
    dz = df[z_col].diff().fillna(0)
    ds = np.sqrt(dx**2 + dy**2 + dz**2)
    arc = ds.rolling(window, min_periods=1).sum()
    x_shift = (df[x_col] - df[x_col].shift(window)).fillna(0)
    y_shift = (df[y_col] - df[y_col].shift(window)).fillna(0)
    z_shift = (df[z_col] - df[z_col].shift(window)).fillna(0)
    chord = np.sqrt(x_shift**2 + y_shift**2 + z_shift**2)
    return (arc / chord.replace(0, np.nan)).fillna(1.0)


def advanced_engineer_features(df, well_col, depth_col, gr_col, x_col, y_col, z_col):
    df = df.copy()
    
    # Per-well GR normalization
    stats = df.groupby(well_col)[gr_col].agg(['mean', 'std']).rename(
        columns={'mean': '_mu', 'std': '_sd'})
    df = df.merge(stats, on=well_col, how='left')
    df['GR_norm'] = (df[gr_col] - df['_mu']) / (df['_sd'] + 1e-6)
    df.drop(columns=['_mu', '_sd'], inplace=True)
    
    all_parts = []
    
    for well_id, gdf in df.groupby(well_col):
        gdf = gdf.sort_values(depth_col).reset_index(drop=True)
        gr  = gdf['GR_norm']
        
        # ── ROLLING STATS ────────────────────────────────────────────────
        for w in [3, 5, 10, 20, 50, 100]:
            r = gr.rolling(w, min_periods=1)
            gdf[f'gr_mean_{w}']   = r.mean()
            gdf[f'gr_std_{w}']    = r.std().fillna(0)
            gdf[f'gr_min_{w}']    = r.min()
            gdf[f'gr_max_{w}']    = r.max()
            gdf[f'gr_range_{w}']  = gdf[f'gr_max_{w}'] - gdf[f'gr_min_{w}']
            gdf[f'gr_skew_{w}']   = r.skew().fillna(0)
            gdf[f'gr_kurt_{w}']   = r.kurt().fillna(0)
            gdf[f'gr_iqr_{w}']    = r.quantile(0.75) - r.quantile(0.25)
        
        # ── EMA + MACD ───────────────────────────────────────────────────
        for span in [3, 5, 10, 20, 50]:
            gdf[f'ema_{span}'] = gr.ewm(span=span, adjust=False).mean()
        for fast, slow in [(3, 10), (5, 20), (10, 50)]:
            macd = gdf[f'ema_{fast}'] - gdf[f'ema_{slow}']
            sig  = macd.ewm(span=9, adjust=False).mean()
            gdf[f'macd_{fast}_{slow}']      = macd
            gdf[f'macd_hist_{fast}_{slow}'] = macd - sig
        
        # ── LAGS & LEADS ─────────────────────────────────────────────────
        for lag in [1, 2, 3, 5, 10, 15, 20, 30]:
            gdf[f'gr_lag_{lag}']  = gr.shift(lag)
            gdf[f'gr_lead_{lag}'] = gr.shift(-lag)
        
        # ── DERIVATIVES ──────────────────────────────────────────────────
        gdf['gr_d1']     = gr.diff(1).fillna(0)
        gdf['gr_d2']     = gr.diff(2).fillna(0)
        gdf['gr_d1_abs'] = gdf['gr_d1'].abs()
        gdf['gr_d1_sq']  = gdf['gr_d1'] ** 2
        # Savitzky-Golay smoothed derivative (smoother than raw diff)
        if len(gdf) >= 11:
            gdf['gr_sg_d1'] = signal.savgol_filter(gr.fillna(gr.median()), 11, 2, deriv=1)
            gdf['gr_sg_d2'] = signal.savgol_filter(gr.fillna(gr.median()), 11, 2, deriv=2)
        
        # ── DWT ──────────────────────────────────────────────────────────
        for wavelet in ['db4', 'sym4']:
            try:
                coeffs = pywt.wavedec(gr.fillna(gr.median()).values, wavelet, level=4)
                for lvl, c in enumerate(coeffs):
                    up = np.interp(np.linspace(0, len(c)-1, len(gdf)),
                                   np.arange(len(c)), c)
                    label = 'A' if lvl == 0 else f'D{lvl}'
                    gdf[f'dwt_{wavelet}_{label}'] = up
            except Exception:
                pass
        
        # ── 3D TRAJECTORY ────────────────────────────────────────────────
        if all(c in gdf.columns for c in [x_col, y_col, z_col]):
            for w in [5, 10, 20]:
                gdf[f'tort_{w}'] = compute_tortuosity(gdf, x_col, y_col, z_col, w)
            
            dx = gdf[x_col].diff().fillna(0)
            dy = gdf[y_col].diff().fillna(0)
            dz = gdf[z_col].diff().fillna(0)
            gdf['ds']  = np.sqrt(dx**2 + dy**2 + dz**2)
            gdf['dip'] = np.degrees(np.arctan2(dz.abs(),
                                    np.sqrt(dx**2 + dy**2) + 1e-6))
            gdf['cumulative_ds'] = gdf['ds'].cumsum()
            
            # Rate of dip change (second derivative of trajectory)
            gdf['dip_d1'] = gdf['dip'].diff().fillna(0)
        
        # ── DEPTH/POSITION ───────────────────────────────────────────────
        depth_range = gdf[depth_col].max() - gdf[depth_col].min() + 1e-6
        gdf['depth_norm'] = (gdf[depth_col] - gdf[depth_col].min()) / depth_range
        gdf['well_pos']   = np.arange(len(gdf)) / len(gdf)
        gdf['n_samples']  = len(gdf)  # well length as global context feature
        
        all_parts.append(gdf)
    
    return pd.concat(all_parts, ignore_index=True)


print('Building advanced features (this may take a few minutes)...')
train_fe = advanced_engineer_features(train, WELL_COL, DEPTH_COL, GR_COL, X_COL, Y_COL, Z_COL)
test_fe  = advanced_engineer_features(test,  WELL_COL, DEPTH_COL, GR_COL, X_COL, Y_COL, Z_COL)

print(f'Features generated: {train_fe.shape[1]} columns  ({train_fe.shape[1] - 10} engineered)')

In [ ]:
# Prepare feature matrix
drop_cols = [TARGET_COL, WELL_COL, 'row_id', 'id']
feature_cols = [c for c in train_fe.columns
                if c not in drop_cols and train_fe[c].dtype != object]

X_train = train_fe[feature_cols].astype(np.float32)
y_train = train_fe[TARGET_COL].astype(np.float32)
groups  = train_fe[WELL_COL]
X_test  = test_fe[feature_cols].astype(np.float32)

medians = X_train.median()
X_train = X_train.fillna(medians)
X_test  = X_test.fillna(medians)

gkf    = GroupKFold(n_splits=N_FOLDS)
splits = list(gkf.split(X_train, y_train, groups=groups))

print(f'Feature matrix: {X_train.shape}  |  {len(feature_cols)} features')

---
# PART 2 — Optuna Hyperparameter Tuning

**What**: Bayesian optimization of model hyperparameters using Optuna's TPE (Tree-structured Parzen Estimator) sampler.

**Why Optuna over grid search**:
- Grid search: O(N^k) evaluations — exponential in number of parameters
- Random search: better, but no learning from past evaluations
- TPE: builds a probabilistic model of which configurations work well and samples from the promising region

**Why not just use defaults**: LightGBM defaults are conservative (31 leaves, high learning rate). Our data has ~40 features and strong sequential structure — we need deeper trees with more regularization.

In [ ]:
def objective_lgb(trial, X, y, groups, splits, n_folds_for_tuning=3):
    """
    Optuna objective for LightGBM.
    Uses only the first n_folds_for_tuning folds for speed.
    """
    params = {
        'objective':         'regression',
        'metric':            'rmse',
        'verbosity':         -1,
        'seed':              SEED,
        # ── Parameters to tune ──────────────────────────────────────────
        'learning_rate':     trial.suggest_float('learning_rate',    0.005, 0.1,  log=True),
        'num_leaves':        trial.suggest_int(  'num_leaves',        31,   511),
        'max_depth':         trial.suggest_int(  'max_depth',          3,    12),
        'min_child_samples': trial.suggest_int(  'min_child_samples',  5,   100),
        'feature_fraction':  trial.suggest_float('feature_fraction',  0.4,   1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction',  0.4,   1.0),
        'bagging_freq':      trial.suggest_int(  'bagging_freq',       1,     7),
        'reg_alpha':         trial.suggest_float('reg_alpha',        1e-8,  10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda',       1e-8,  10.0, log=True),
        'min_split_gain':    trial.suggest_float('min_split_gain',   0.0,    1.0),
    }
    
    oof = np.zeros(len(X))
    
    for fold_i, (tr_idx, val_idx) in enumerate(splits[:n_folds_for_tuning]):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)
        
        model = lgb.train(
            params, dtrain,
            num_boost_round=2000,
            valid_sets=[dval],
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(period=-1)
            ]
        )
        oof[val_idx] = model.predict(X_val)
        
        # Pruning: stop unpromising trials early
        fold_rmse = np.sqrt(mean_squared_error(y_val, oof[val_idx]))
        trial.report(fold_rmse, fold_i)
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    used_idx = np.concatenate([v for _, v in splits[:n_folds_for_tuning]])
    return np.sqrt(mean_squared_error(y.iloc[used_idx], oof[used_idx]))


print('Starting Optuna study for LightGBM (50 trials)...')
print('Each trial takes ~30–60s depending on CPU.')

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1)
)

study.optimize(
    lambda trial: objective_lgb(trial, X_train, y_train, groups, splits),
    n_trials=50,
    show_progress_bar=True
)

print(f'\nBest trial RMSE : {study.best_value:.4f}')
print(f'Best params     : {study.best_params}')

In [ ]:
# Visualize Optuna optimization history
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_contour
)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Trial history
trial_values = [t.value for t in study.trials if t.value is not None]
best_so_far  = np.minimum.accumulate(trial_values)
axes[0].plot(trial_values, 'o', alpha=0.5, markersize=4, label='Trial RMSE')
axes[0].plot(best_so_far, lw=2, color='red', label='Best so far')
axes[0].set_title('Optuna Optimization History')
axes[0].set_xlabel('Trial #')
axes[0].set_ylabel('OOF RMSE')
axes[0].legend()

# Parameter importances
importances = optuna.importance.get_param_importances(study)
params_sorted = sorted(importances.items(), key=lambda x: x[1], reverse=True)
names, vals = zip(*params_sorted)
axes[1].barh(list(names)[::-1], list(vals)[::-1], color='steelblue')
axes[1].set_title('Hyperparameter Importances (Optuna FAnova)')
axes[1].set_xlabel('Relative importance')

plt.tight_layout()
plt.show()

In [ ]:
# Train final LGB with best params (lower LR + more rounds for full train)
best_lgb_params = {
    **study.best_params,
    'objective':  'regression',
    'metric':     'rmse',
    'verbosity':  -1,
    'seed':       SEED,
    'learning_rate': study.best_params['learning_rate'] * 0.5,  # halve LR
}

lgb_oof_preds  = np.zeros(len(X_train))
lgb_test_preds = np.zeros(len(X_test))

for fold_i, (tr_idx, val_idx) in enumerate(splits):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    
    model = lgb.train(
        best_lgb_params, dtrain,
        num_boost_round=6000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(500)]
    )
    lgb_oof_preds[val_idx] = model.predict(X_val)
    lgb_test_preds += model.predict(X_test) / N_FOLDS
    fold_rmse = np.sqrt(mean_squared_error(y_val, lgb_oof_preds[val_idx]))
    print(f'  Fold {fold_i}: {fold_rmse:.4f}')

lgb_rmse = np.sqrt(mean_squared_error(y_train, lgb_oof_preds))
print(f'\nTuned LightGBM OOF RMSE: {lgb_rmse:.4f}')

---
# PART 3 — Stacking Meta-Learner

**What**: Train a second-level model (meta-learner) that takes as input the OOF predictions of the base models and learns their optimal combination.

**Why it beats simple weighted averaging**: The meta-learner can discover non-linear relationships between the base models. For example: "trust LGB when GR is low, trust XGB when tortuosity is high".

**Warning**: Stacking is prone to overfitting. Always use OOF predictions (never in-fold predictions) as meta-features.

In [ ]:
# We need OOF from at least 2 other models. We'll use XGB + CatBoost here.
# (Run their training first — same pattern as 02_modeling.ipynb)

# --- XGBoost OOF (run this block) ---
xgb_oof_preds  = np.zeros(len(X_train))
xgb_test_preds = np.zeros(len(X_test))

xgb_params = {
    'objective': 'reg:squarederror', 'eval_metric': 'rmse',
    'learning_rate': 0.02, 'max_depth': 8,
    'min_child_weight': 15, 'subsample': 0.8,
    'colsample_bytree': 0.7, 'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'n_estimators': 4000, 'early_stopping_rounds': 100,
    'random_state': SEED, 'n_jobs': -1, 'verbosity': 0,
}

for fold_i, (tr_idx, val_idx) in enumerate(splits):
    m = xgb.XGBRegressor(**xgb_params)
    m.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx],
          eval_set=[(X_train.iloc[val_idx], y_train.iloc[val_idx])], verbose=500)
    xgb_oof_preds[val_idx]  = m.predict(X_train.iloc[val_idx])
    xgb_test_preds += m.predict(X_test) / N_FOLDS
    print(f'  XGB Fold {fold_i}: {np.sqrt(mean_squared_error(y_train.iloc[val_idx], xgb_oof_preds[val_idx])):.4f}')

xgb_rmse = np.sqrt(mean_squared_error(y_train, xgb_oof_preds))
print(f'XGBoost OOF RMSE: {xgb_rmse:.4f}')

In [ ]:
# --- CatBoost OOF ---
cat_oof_preds  = np.zeros(len(X_train))
cat_test_preds = np.zeros(len(X_test))

for fold_i, (tr_idx, val_idx) in enumerate(splits):
    m = CatBoostRegressor(
        iterations=4000, learning_rate=0.02, depth=8,
        l2_leaf_reg=5, loss_function='RMSE',
        early_stopping_rounds=100, random_seed=SEED, verbose=500)
    m.fit(Pool(X_train.iloc[tr_idx], y_train.iloc[tr_idx]),
          eval_set=Pool(X_train.iloc[val_idx], y_train.iloc[val_idx]))
    cat_oof_preds[val_idx]  = m.predict(X_train.iloc[val_idx])
    cat_test_preds += m.predict(X_test) / N_FOLDS
    print(f'  CAT Fold {fold_i}: {np.sqrt(mean_squared_error(y_train.iloc[val_idx], cat_oof_preds[val_idx])):.4f}')

cat_rmse = np.sqrt(mean_squared_error(y_train, cat_oof_preds))
print(f'CatBoost OOF RMSE: {cat_rmse:.4f}')

In [ ]:
# Build meta-feature matrix: OOF predictions + original features subset
# We include a few "context" features to let the meta-learner be input-aware
meta_context_cols = ['depth_norm', 'well_pos', 'GR_norm',
                     'gr_d1', 'tort_10'] 
meta_context_cols = [c for c in meta_context_cols if c in X_train.columns]

meta_train = np.column_stack([
    lgb_oof_preds, xgb_oof_preds, cat_oof_preds,
    X_train[meta_context_cols].values
])
meta_test = np.column_stack([
    lgb_test_preds, xgb_test_preds, cat_test_preds,
    X_test[meta_context_cols].values
])

meta_col_names = ['lgb', 'xgb', 'cat'] + meta_context_cols
print(f'Meta-feature matrix: {meta_train.shape}')


# Meta-learner: Ridge regression (simple, low variance, prevents overfitting)
# We use GroupKFold again to avoid leakage
meta_oof  = np.zeros(len(X_train))
meta_test_preds = np.zeros(len(X_test))

scaler = StandardScaler()

for fold_i, (tr_idx, val_idx) in enumerate(splits):
    m_tr  = scaler.fit_transform(meta_train[tr_idx])
    m_val = scaler.transform(meta_train[val_idx])
    m_te  = scaler.transform(meta_test)
    
    meta_model = Ridge(alpha=10.0)
    meta_model.fit(m_tr, y_train.iloc[tr_idx])
    meta_oof[val_idx] = meta_model.predict(m_val)
    meta_test_preds  += meta_model.predict(m_te) / N_FOLDS

stack_rmse = np.sqrt(mean_squared_error(y_train, meta_oof))
print(f'\nStacking OOF RMSE: {stack_rmse:.4f}')
print(f'  vs LGB alone   : {lgb_rmse:.4f}')
print(f'  vs XGB alone   : {xgb_rmse:.4f}')
print(f'  vs CAT alone   : {cat_rmse:.4f}')

# Meta-learner coefficients
print(f'\nMeta-learner coefficients:')
for name, coef in zip(meta_col_names, meta_model.coef_):
    print(f'  {name:30s}: {coef:+.4f}')

---
# PART 4 — LSTM Sequence Model

**What**: A bidirectional LSTM that treats the GR log as a time series and predicts TVT at each depth point.

**Why tree models miss this**: LightGBM sees each row independently — it learns *which feature values predict TVT* but not *how TVT evolves along the well*. An LSTM explicitly models the sequential dependency: it reads the full GR history before predicting.

**Bidirectional**: we use both past and future GR context, which is valid because the full horizontal well log is available at inference time (this is not a true forecasting problem).

**Architecture**: `BiLSTM(128) → Dropout(0.2) → BiLSTM(64) → Dense(32, relu) → Dense(1)`

In [ ]:
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    TORCH_AVAILABLE = True
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'PyTorch available. Device: {device}')
except ImportError:
    TORCH_AVAILABLE = False
    print('PyTorch not installed. Run: pip install torch')
    print('LSTM cell below will show the architecture but not execute.')

In [ ]:
if TORCH_AVAILABLE:
    
    class WellDataset(Dataset):
        """
        Each item is one complete well's feature sequence.
        Sequences are padded to the same length within each batch.
        """
        def __init__(self, df, feature_cols, target_col, well_col, depth_col):
            self.sequences = []
            self.targets   = []
            self.lengths   = []
            
            for _, gdf in df.groupby(well_col):
                gdf = gdf.sort_values(depth_col)
                X   = gdf[feature_cols].fillna(0).values.astype(np.float32)
                y   = gdf[target_col].values.astype(np.float32) if target_col in gdf else None
                self.sequences.append(torch.tensor(X))
                if y is not None:
                    self.targets.append(torch.tensor(y))
                self.lengths.append(len(X))
        
        def __len__(self):
            return len(self.sequences)
        
        def __getitem__(self, idx):
            item = {'X': self.sequences[idx], 'length': self.lengths[idx]}
            if self.targets:
                item['y'] = self.targets[idx]
            return item
    
    
    def collate_fn(batch):
        """Pad variable-length well sequences to the same length in the batch."""
        from torch.nn.utils.rnn import pad_sequence
        Xs  = [item['X'] for item in batch]
        Xs_padded = pad_sequence(Xs, batch_first=True, padding_value=0.0)
        lengths = torch.tensor([item['length'] for item in batch])
        out = {'X': Xs_padded, 'lengths': lengths}
        if 'y' in batch[0]:
            ys = [item['y'] for item in batch]
            out['y'] = pad_sequence(ys, batch_first=True, padding_value=0.0)
        return out
    
    
    class BiLSTMRegressor(nn.Module):
        """
        Bidirectional LSTM for TVT prediction along wellbore sequences.
        
        Forward pass:
          Input:  (batch, seq_len, n_features)
          Output: (batch, seq_len, 1)  — TVT at each depth point
        """
        def __init__(self, n_features, hidden_size=128, n_layers=2, dropout=0.2):
            super().__init__()
            
            # Input projection
            self.input_proj = nn.Sequential(
                nn.Linear(n_features, hidden_size),
                nn.LayerNorm(hidden_size),
                nn.ReLU()
            )
            
            # Bidirectional LSTM stack
            self.lstm = nn.LSTM(
                input_size=hidden_size,
                hidden_size=hidden_size,
                num_layers=n_layers,
                batch_first=True,
                bidirectional=True,
                dropout=dropout if n_layers > 1 else 0.0
            )
            
            # Output head
            self.head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(hidden_size * 2, 64),  # *2 for bidirectional
                nn.ReLU(),
                nn.Linear(64, 1)
            )
        
        def forward(self, x, lengths=None):
            x = self.input_proj(x)
            
            if lengths is not None:
                from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
                packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True,
                                              enforce_sorted=False)
                out, _ = self.lstm(packed)
                out, _ = pad_packed_sequence(out, batch_first=True)
            else:
                out, _ = self.lstm(x)
            
            return self.head(out).squeeze(-1)  # (batch, seq_len)
    
    
    # Use a subset of features for LSTM (avoid redundancy, keep physical features)
    lstm_feature_cols = [
        'GR_norm', 'gr_mean_5', 'gr_mean_20', 'gr_std_10',
        'gr_d1', 'gr_d2', 'gr_sg_d1', 'gr_sg_d2',
        'ema_5', 'ema_20', 'macd_5_20', 'macd_hist_5_20',
        'dwt_db4_A', 'dwt_db4_D1', 'dwt_db4_D2',
        'depth_norm', 'well_pos',
        'tort_10', 'dip', 'dip_d1'
    ]
    lstm_feature_cols = [c for c in lstm_feature_cols if c in X_train.columns]
    
    model_lstm = BiLSTMRegressor(
        n_features=len(lstm_feature_cols),
        hidden_size=128, n_layers=2, dropout=0.2
    ).to(device)
    
    total_params = sum(p.numel() for p in model_lstm.parameters() if p.requires_grad)
    print(f'BiLSTM model — {total_params:,} trainable parameters')
    print(model_lstm)

In [ ]:
if TORCH_AVAILABLE:
    
    def train_lstm_fold(train_df, val_df, test_df, feature_cols, target_col,
                        well_col, depth_col, n_epochs=30, lr=1e-3, batch_size=4):
        """
        Train one fold of BiLSTM. Returns (oof_preds_array, test_preds_array).
        """
        train_ds = WellDataset(train_df, feature_cols, target_col, well_col, depth_col)
        val_ds   = WellDataset(val_df,   feature_cols, target_col, well_col, depth_col)
        test_ds  = WellDataset(test_df,  feature_cols, None,       well_col, depth_col)
        
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn)
        val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
        test_loader  = DataLoader(test_ds,  batch_size=1,          shuffle=False, collate_fn=collate_fn)
        
        model = BiLSTMRegressor(len(feature_cols), 128, 2, 0.2).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
        criterion = nn.HuberLoss(delta=1.5)  # robust to TVT outliers
        
        best_val_rmse = float('inf')
        best_state = None
        
        for epoch in range(n_epochs):
            # Train
            model.train()
            for batch in train_loader:
                X = batch['X'].to(device)
                y = batch['y'].to(device)
                lengths = batch['lengths']
                optimizer.zero_grad()
                pred = model(X, lengths)
                # Mask padding positions
                mask = torch.zeros_like(pred, dtype=torch.bool)
                for i, l in enumerate(lengths):
                    mask[i, :l] = True
                loss = criterion(pred[mask], y[mask])
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()
            
            # Validate
            model.eval()
            val_preds, val_targets = [], []
            with torch.no_grad():
                for batch in val_loader:
                    X = batch['X'].to(device)
                    lengths = batch['lengths']
                    pred = model(X, lengths)
                    for i, l in enumerate(lengths.tolist()):
                        val_preds.append(pred[i, :l].cpu().numpy())
                        val_targets.append(batch['y'][i, :l].numpy())
            
            val_preds_flat   = np.concatenate(val_preds)
            val_targets_flat = np.concatenate(val_targets)
            val_rmse = np.sqrt(mean_squared_error(val_targets_flat, val_preds_flat))
            
            if val_rmse < best_val_rmse:
                best_val_rmse = val_rmse
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            
            if (epoch + 1) % 5 == 0:
                print(f'    Epoch {epoch+1:3d}  val RMSE: {val_rmse:.4f}  (best: {best_val_rmse:.4f})')
        
        # Predict with best model
        model.load_state_dict(best_state)
        model.eval()
        test_preds = []
        with torch.no_grad():
            for batch in test_loader:
                X = batch['X'].to(device)
                lengths = batch['lengths']
                pred = model(X, lengths)
                for i, l in enumerate(lengths.tolist()):
                    test_preds.append(pred[i, :l].cpu().numpy())
        
        return val_preds_flat, np.concatenate(test_preds), best_val_rmse
    
    
    print('LSTM training function defined. Run the next cell to train all folds.')
    print('Expected time: ~10–30 min on CPU, ~3–5 min on GPU per fold.')

In [ ]:
# Uncomment to run full LSTM training
# lstm_oof_preds  = np.zeros(len(train_fe))
# lstm_test_preds_list = []
#
# for fold_i, (tr_idx, val_idx) in enumerate(splits):
#     print(f'Fold {fold_i}...')
#     tr_df  = train_fe.iloc[tr_idx]
#     val_df = train_fe.iloc[val_idx]
#
#     val_pred, test_pred, best_rmse = train_lstm_fold(
#         tr_df, val_df, test_fe,
#         lstm_feature_cols, TARGET_COL,
#         WELL_COL, DEPTH_COL,
#         n_epochs=50, lr=1e-3, batch_size=4
#     )
#     lstm_oof_preds[val_idx] = val_pred
#     lstm_test_preds_list.append(test_pred)
#     print(f'  Fold {fold_i} best RMSE: {best_rmse:.4f}')
#
# lstm_test_preds = np.mean(lstm_test_preds_list, axis=0)
# lstm_rmse = np.sqrt(mean_squared_error(y_train, lstm_oof_preds))
# print(f'BiLSTM OOF RMSE: {lstm_rmse:.4f}')

print('LSTM training block commented out for quick execution.')
print('Uncomment to train. Expected LB improvement: -0.5 to -1.5 RMSE over LGB alone.')

---
# PART 5 — Particle Filter Post-Processing

**What**: Apply a particle filter (Sequential Monte Carlo) to smooth and physically constrain the TVT predictions.

**Why it works**: TVT cannot change arbitrarily — geology is continuous and smooth. The ML model may produce noisy spike predictions at isolated depth points. A particle filter:
1. Maintains a distribution of possible TVT states at each depth
2. Propagates states forward using a **motion model** (TVT changes slowly)
3. Updates states using the ML prediction as a **measurement**
4. Outputs the smoothed, physically consistent TVT trajectory

**This is the closest technique to how real geosteering software works** — ROGII's own system uses similar Bayesian filtering.

In [ ]:
def particle_filter_smooth(raw_predictions, n_particles=500,
                           process_noise=0.5, measurement_noise=2.0,
                           seed=SEED):
    """
    Particle filter for TVT trajectory smoothing.
    
    Model:
      State transition: TVT_t = TVT_{t-1} + N(0, process_noise²)
                        (geology changes slowly along depth)
      Measurement:      y_t   = TVT_t + N(0, measurement_noise²)
                        (ML prediction is noisy observation of true TVT)
    
    Parameters:
      process_noise    : how fast TVT can change per depth step (meters)
                         smaller → smoother output
      measurement_noise: how much we trust the ML prediction vs smoothness
                         larger → smoother output, less faithful to raw ML
    
    Returns smoothed TVT array of same length as raw_predictions.
    """
    rng = np.random.default_rng(seed)
    n = len(raw_predictions)
    
    # Initialize particles around first prediction
    particles = rng.normal(raw_predictions[0], measurement_noise, n_particles)
    weights   = np.ones(n_particles) / n_particles
    
    smoothed = np.zeros(n)
    
    for t in range(n):
        # ── PREDICT (motion model) ───────────────────────────────────────
        particles += rng.normal(0, process_noise, n_particles)
        
        # ── UPDATE (measurement) ─────────────────────────────────────────
        # Likelihood: how well does each particle explain the ML measurement?
        diff = particles - raw_predictions[t]
        log_likelihoods = -0.5 * (diff / measurement_noise) ** 2
        log_likelihoods -= log_likelihoods.max()  # numerical stability
        weights = np.exp(log_likelihoods)
        weights /= weights.sum()
        
        # ── ESTIMATE ──────────────────────────────────────────────────────
        smoothed[t] = np.average(particles, weights=weights)
        
        # ── RESAMPLE (avoid particle degeneracy) ──────────────────────────
        # Effective sample size: when ESS drops below n/2, resample
        ess = 1.0 / (weights ** 2).sum()
        if ess < n_particles / 2:
            indices  = rng.choice(n_particles, size=n_particles, p=weights)
            particles = particles[indices]
            weights   = np.ones(n_particles) / n_particles
    
    return smoothed


def apply_particle_filter_per_well(preds_array, well_ids, process_noise=0.5,
                                    measurement_noise=2.0, n_particles=500):
    """
    Apply particle filter independently to each well's TVT sequence.
    Wells must be sorted by depth before calling this.
    """
    smoothed = preds_array.copy()
    for well_id in np.unique(well_ids):
        mask = well_ids == well_id
        smoothed[mask] = particle_filter_smooth(
            preds_array[mask],
            n_particles=n_particles,
            process_noise=process_noise,
            measurement_noise=measurement_noise
        )
    return smoothed


# Demo: visualize particle filter on a sample well
sample_well_mask = (train_fe[WELL_COL] == train_fe[WELL_COL].unique()[0])
raw_tvt  = lgb_oof_preds[sample_well_mask][:200]
true_tvt = y_train.values[sample_well_mask][:200]

# Test different smoothing strengths
smooth_tight  = particle_filter_smooth(raw_tvt, process_noise=0.3, measurement_noise=1.5)
smooth_medium = particle_filter_smooth(raw_tvt, process_noise=0.5, measurement_noise=2.5)
smooth_loose  = particle_filter_smooth(raw_tvt, process_noise=1.0, measurement_noise=4.0)

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

depth_idx = np.arange(len(raw_tvt))
axes[0].plot(depth_idx, true_tvt,      color='black',      lw=2,   label='True TVT', zorder=5)
axes[0].plot(depth_idx, raw_tvt,       color='grey',       lw=0.8, alpha=0.7, label='Raw ML prediction')
axes[0].plot(depth_idx, smooth_tight,  color='steelblue',  lw=1.5, label='PF tight (proc=0.3, meas=1.5)')
axes[0].plot(depth_idx, smooth_medium, color='darkorange', lw=1.5, label='PF medium (proc=0.5, meas=2.5)')
axes[0].plot(depth_idx, smooth_loose,  color='green',      lw=1.5, label='PF loose (proc=1.0, meas=4.0)')
axes[0].set_title('Particle Filter Smoothing of TVT Predictions')
axes[0].set_ylabel('TVT (m)')
axes[0].legend(fontsize=8)

def rmse(a, b): return np.sqrt(mean_squared_error(a, b))
axes[1].bar(['Raw ML', 'PF tight', 'PF medium', 'PF loose'],
            [rmse(true_tvt, raw_tvt), rmse(true_tvt, smooth_tight),
             rmse(true_tvt, smooth_medium), rmse(true_tvt, smooth_loose)],
            color=['grey', 'steelblue', 'darkorange', 'green'])
axes[1].set_title('RMSE comparison')
axes[1].set_ylabel('RMSE (m)')

plt.tight_layout()
plt.show()

---
# PART 6 — Pseudo-Labeling

**What**: Use the model's confident test-set predictions as additional training data, then retrain.

**Why it works**: The test wells share the same geological basin as training wells. If the model predicts TVT=12.3m with high confidence for a test point, that prediction is likely correct enough to serve as a training label.

**Risk**: If initial model quality is poor, pseudo-labels are noisy and degrade performance. Always validate the improvement.

**Confidence filtering**: Only use test samples where `|pred_std across CV folds| < threshold` — high agreement across folds → high confidence.

In [ ]:
def pseudo_label_with_confidence(test_fe, fold_test_preds_list, target_col,
                                  std_threshold=2.0, sample_weight=0.3):
    """
    Generate pseudo-labels for test samples where all CV fold predictions agree.
    
    Parameters:
      fold_test_preds_list: list of arrays, one prediction per fold
      std_threshold:        max allowed std across fold predictions (meters)
      sample_weight:        weight for pseudo-labeled samples vs real training (0–1)
    
    Returns:
      pseudo_df: test rows with pseudo-label column
      weights:   sample weights array (for lgb/xgb `sample_weight` parameter)
    """
    preds_matrix = np.column_stack(fold_test_preds_list)
    pred_mean = preds_matrix.mean(axis=1)
    pred_std  = preds_matrix.std(axis=1)
    
    confident_mask = pred_std < std_threshold
    n_confident = confident_mask.sum()
    print(f'Confident test samples: {n_confident}/{len(test_fe)} '
          f'({n_confident/len(test_fe)*100:.1f}%) with std < {std_threshold}m')
    
    pseudo_df = test_fe[confident_mask].copy()
    pseudo_df[target_col] = pred_mean[confident_mask]
    pseudo_weights = np.full(n_confident, sample_weight)
    
    return pseudo_df, pseudo_weights


# Example usage (run after main LGB training across all folds)
# Collect per-fold test predictions (not averaged)

print('Pseudo-labeling function defined.')
print('Usage after main training loop:')
print('  pseudo_df, pw = pseudo_label_with_confidence(test_fe, fold_preds_list, TARGET_COL, std_threshold=2.0)')
print('  train_aug = pd.concat([train_fe, pseudo_df], ignore_index=True)')
print('  # Retrain with sample_weight parameter in LGB')

---
# PART 7 — Test-Time Augmentation (TTA)

**What**: At inference time, make multiple slightly perturbed predictions and average them.

**Why it works**: Tree models are deterministic but sensitive to input perturbations. Small GR noise ≈ the natural measurement uncertainty of real logging tools. Averaging over perturbations gives a prediction that is more robust to this noise.

**Perturbations to try**:
- Add Gaussian noise to GR (σ = 2–5% of GR std)
- Reverse the depth sequence (predict from the other end)
- Add small depth offsets (shift the depth axis ±0.5m)


In [ ]:
def tta_predict(models_list, X_test_orig, train_fe, test_fe, n_aug=5, noise_scale=0.02):
    """
    Test-Time Augmentation for tree models.
    
    models_list: list of trained LGB/XGB models (one per fold or all)
    noise_scale: fraction of feature std to use as perturbation noise
    """
    # Compute feature stds from training data for noise calibration
    feat_stds = X_test_orig.std()
    
    all_preds = []
    
    # Original prediction
    for model in models_list:
        all_preds.append(model.predict(X_test_orig))
    
    # Augmented predictions
    rng = np.random.default_rng(SEED)
    for aug_i in range(n_aug - 1):
        X_aug = X_test_orig.copy()
        # Add independent noise to each feature
        noise = rng.normal(0, 1, X_aug.shape).astype(np.float32) \
                * (feat_stds.values * noise_scale)
        X_aug += noise
        
        for model in models_list:
            all_preds.append(model.predict(X_aug))
    
    return np.mean(all_preds, axis=0), np.std(all_preds, axis=0)


print('TTA function defined. Expected improvement: -0.1 to -0.3 RMSE.')
print('Larger n_aug (10-20) gives diminishing returns after ~5 augmentations.')

---
# PART 8 — Final Ensemble with All Techniques

In [ ]:
from scipy.optimize import minimize

def build_final_submission(oof_dict, test_dict, y_train, test_fe,
                            sample_sub, well_col, depth_col,
                            use_particle_filter=True):
    """
    Combines all OOF predictions into a final submission.
    
    oof_dict  : {'lgb': array, 'xgb': array, 'cat': array, ...}
    test_dict : same keys, test predictions
    """
    model_names = list(oof_dict.keys())
    oof_matrix  = np.column_stack([oof_dict[k]  for k in model_names])
    test_matrix = np.column_stack([test_dict[k] for k in model_names])
    y_true = y_train.values
    
    # Optimize weights
    def neg_rmse(weights):
        w = np.abs(weights) / np.abs(weights).sum()
        return np.sqrt(mean_squared_error(y_true, oof_matrix @ w))
    
    res = minimize(neg_rmse, np.ones(len(model_names)) / len(model_names),
                   method='Nelder-Mead', options={'maxiter': 5000, 'xatol': 1e-7})
    best_w = np.abs(res.x) / np.abs(res.x).sum()
    
    ensemble_oof  = oof_matrix  @ best_w
    ensemble_test = test_matrix @ best_w
    
    print('Optimal weights:')
    for name, w in zip(model_names, best_w):
        print(f'  {name:15s}: {w:.4f}')
    print(f'Ensemble OOF RMSE: {np.sqrt(mean_squared_error(y_true, ensemble_oof)):.4f}')
    
    # Particle filter smoothing per well
    if use_particle_filter:
        test_well_ids = test_fe[well_col].values
        ensemble_test = apply_particle_filter_per_well(
            ensemble_test, test_well_ids,
            process_noise=0.5, measurement_noise=2.5, n_particles=300
        )
        print('Particle filter applied to test predictions.')
    
    # Build submission
    id_col   = sample_sub.columns[0]
    pred_col = sample_sub.columns[1]
    
    sub = pd.DataFrame({
        id_col:   sample_sub[id_col].values,
        pred_col: ensemble_test
    })
    
    return sub, ensemble_oof


# Collect all available predictions
oof_dict  = {'lgb': lgb_oof_preds, 'xgb': xgb_oof_preds, 'cat': cat_oof_preds}
test_dict = {'lgb': lgb_test_preds, 'xgb': xgb_test_preds, 'cat': cat_test_preds}

# Add LSTM if available
# if TORCH_AVAILABLE and 'lstm_oof_preds' in dir():
#     oof_dict['lstm']  = lstm_oof_preds
#     test_dict['lstm'] = lstm_test_preds

submission, final_oof = build_final_submission(
    oof_dict, test_dict, y_train, test_fe,
    sample_sub, WELL_COL, DEPTH_COL,
    use_particle_filter=True
)

sub_path = SUBMIT_DIR / 'submission_advanced.csv'
submission.to_csv(sub_path, index=False)
print(f'\nSubmission saved: {sub_path}')
print(submission.head())

---
# PART 9 — Diagnostics & Calibration

In [ ]:
# Error analysis: where does the model fail?
train_diag = train_fe[[WELL_COL, DEPTH_COL, TARGET_COL, GR_COL, 'depth_norm']].copy()
train_diag['pred']     = final_oof
train_diag['residual'] = train_diag[TARGET_COL] - train_diag['pred']
train_diag['abs_err']  = train_diag['residual'].abs()

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.35)

# 1. Error vs GR
ax = fig.add_subplot(gs[0, 0])
ax.hexbin(train_diag[GR_COL], train_diag['abs_err'], gridsize=40, cmap='YlOrRd', mincnt=1)
ax.set_title('Abs error vs GR value')
ax.set_xlabel('GR (API)')
ax.set_ylabel('|Error| (m)')

# 2. Error vs TVT (target)
ax = fig.add_subplot(gs[0, 1])
ax.hexbin(train_diag[TARGET_COL], train_diag['abs_err'], gridsize=40, cmap='YlOrRd', mincnt=1)
ax.set_title('Abs error vs true TVT')
ax.set_xlabel('True TVT (m)')

# 3. Error vs depth position within well
ax = fig.add_subplot(gs[0, 2])
ax.hexbin(train_diag['depth_norm'], train_diag['abs_err'], gridsize=40, cmap='YlOrRd', mincnt=1)
ax.set_title('Abs error vs depth position')
ax.set_xlabel('Relative depth (0=top, 1=bottom)')

# 4. Per-well RMSE sorted
well_rmse = train_diag.groupby(WELL_COL)['abs_err'].apply(
    lambda x: np.sqrt((x**2).mean())).sort_values(ascending=False)

ax = fig.add_subplot(gs[1, :])
ax.bar(range(len(well_rmse)), well_rmse.values, color='steelblue', alpha=0.7)
ax.axhline(well_rmse.median(), color='red', ls='--', lw=2,
           label=f'Median RMSE: {well_rmse.median():.2f}m')
ax.set_title('Per-Well RMSE (sorted descending)')
ax.set_xlabel('Well rank')
ax.set_ylabel('RMSE (m)')
ax.legend()

# 5. Residual distribution
ax = fig.add_subplot(gs[2, 0])
ax.hist(train_diag['residual'], bins=100, color='steelblue', edgecolor='none', density=True)
from scipy.stats import norm
x = np.linspace(train_diag['residual'].min(), train_diag['residual'].max(), 200)
ax.plot(x, norm.pdf(x, train_diag['residual'].mean(), train_diag['residual'].std()),
        color='red', lw=2, label='Normal fit')
ax.set_title('Residual distribution (heavy tails = outlier wells)')
ax.legend()

# 6. Quantile-quantile
from scipy.stats import probplot
ax = fig.add_subplot(gs[2, 1])
(osm, osr), (slope, intercept, _) = probplot(train_diag['residual'].dropna())
ax.plot(osm, osr, '.', markersize=2, alpha=0.5)
ax.plot(osm, slope * np.array(osm) + intercept, 'r-', lw=2)
ax.set_title('Q-Q plot of residuals')
ax.set_xlabel('Theoretical quantiles')
ax.set_ylabel('Sample quantiles')

# 7. Predicted vs actual
ax = fig.add_subplot(gs[2, 2])
sample_n = min(5000, len(train_diag))
idx = np.random.choice(len(train_diag), sample_n, replace=False)
ax.scatter(train_diag[TARGET_COL].iloc[idx], train_diag['pred'].iloc[idx],
           alpha=0.2, s=3, c=train_diag['abs_err'].iloc[idx], cmap='RdYlGn_r')
lo, hi = train_diag[TARGET_COL].min(), train_diag[TARGET_COL].max()
ax.plot([lo, hi], [lo, hi], 'r--', lw=2)
ax.set_title('Predicted vs Actual TVT (color = error)')
ax.set_xlabel('Actual')
ax.set_ylabel('Predicted')

plt.suptitle('Model Diagnostics', fontsize=16, y=1.01)
plt.show()

---
# PART 10 — Roadmap to Top Tier

```
Current trajectory with this notebook:

  Baseline LGB (02_modeling)         ≈ 12–15 RMSE
  + Advanced FE (DWT, MACD, moments) ≈ 10–12
  + Optuna tuning                    ≈  9–11
  + Ensemble (LGB+XGB+CAT)           ≈  8.5–10
  + Stacking meta-learner            ≈  8–9.5
  + Particle filter smoothing        ≈  7.5–9
  + BiLSTM in ensemble               ≈  7–8.5
  + Reference well xcorr features   ≈  6.5–8
  + Pseudo-labeling                  ≈  6–7.5
  ──────────────────────────────────────────
  TOP TIER                           ≈  5–7 RMSE

What separates top-3 from top-20:
  1. Reference well cross-correlation implemented correctly
  2. Particle filter with well-calibrated noise parameters (per-well)
  3. Transformer architecture instead of LSTM (more parallelizable, better long-range)
  4. Geological constraints: TVT monotonicity regions (cannot go up-down-up in tight geology)
  5. Ensemble diversity: include a CNN-1D that looks at raw GR patches
```

### The exotic / underexplored approach

**Treat this as a registration problem** (not a regression problem):

The GR log of the horizontal well is a "shifted and stretched" version of the reference well GR. Finding the TVT is equivalent to finding the optimal **depth registration** (alignment) between the two logs. This is exactly what Dynamic Time Warping (DTW) solves:

```python
from dtaidistance import dtw

# Find optimal alignment between horizontal GR and reference GR
path = dtw.warping_path(horizontal_gr, reference_gr)
# path gives: for each horizontal depth index → best matching reference depth
# TVT = depth difference between matched positions
```

DTW-based TVT estimation is **zero-shot** (no training required) and physically interpretable. It can be used as:
1. A standalone predictor (strong baseline)
2. A feature for the ML ensemble (DTW-estimated TVT as input)
3. A post-processing constraint (final prediction must not deviate too far from DTW estimate)